# Setup Inicial do Ambiente
## Arquitetura Agnóstica (Cloud vs Local)

Este projeto foi construído para ser **100% agnóstico de infraestrutura**. Isso significa que a mesma regra de negócio desenvolvida em PySpark pode ser executada sem nenhuma alteração em:

1. **Nuvem Corporativa (Ex: Databricks, AWS e Azure):** Usando o Unity Catalog, Glue.
2. **Ambiente Local (Docker / Jupyter):** Salvando os arquivos localmente usando o formato aberto Delta Lake (`.save()`).

A variável `ENVIRONMENT` controla essa chave de forma transparente para as camadas Bronze, Silver e Gold. Altere-a dependendo da sua stack.

In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, IntegerType, BooleanType
)

spark = SparkSession.builder.appName("FootballPipeline").getOrCreate()

ENVIRONMENT = os.getenv("SPARK_ENV", "databricks")

if ENVIRONMENT == "databricks":
    PATH_RAW    = "/tmp/football_raw"
    PATH_BRONZE = "/Volumes/workspace/bronze/football_raw"
    PATH_SILVER = "/Volumes/workspace/silver/football_data"
    PATH_GOLD   = "/Volumes/workspace/gold/football_data"
    
    CATALOG_BRONZE = "workspace.bronze"
    CATALOG_SILVER = "workspace.silver"
    CATALOG_GOLD   = "workspace.gold"
else:
    PATH_RAW    = "./data/raw"
    PATH_BRONZE = "./data/bronze"
    PATH_SILVER = "./data/silver"
    PATH_GOLD   = "./data/gold"
    
    CATALOG_BRONZE = "bronze"
    CATALOG_SILVER = "silver"
    CATALOG_GOLD   = "gold"

if ENVIRONMENT == "databricks":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_BRONZE};")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_BRONZE}.football_raw;")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SILVER};")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_GOLD};")

print(f"Ambiente configurado: {ENVIRONMENT}")

Ambiente configurado: databricks
